# Ortofotomapa skośna: demo

In [ ]:
%config InlineBackend.figure_formats = ["jpeg"]
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pil_kwargs": {"quality": 88}}

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from PIL import Image

from ortofoto import WYNIKI
from ortofoto.arkusze import ELEWACJE, KIERUNKI, KOLEJNOSC, NA, czytaj, opis_arkuszy, podklad, zasieg
from ortofoto.budynki import opis, wczytaj_budynki, wybierz
from ortofoto.wycinki import K, dach, dostepne_kierunki, eksport_budynku, elewacja, kadr, okres_kondygnacji

plt.rcParams.update({"figure.dpi": 100, "font.size": 9, "axes.titlesize": 10, "figure.titlesize": 12})
PODSTAWA, DACH = "#ffd60a", "#ff2d55"


def obrys(ax, geom, **kw):
    ax.plot(*geom.exterior.xy, lw=kw.pop("lw", 1.8), **kw)


def skala(ax, metry, loc="lower right"):
    pasek = AnchoredSizeBar(ax.transData, metry, f"{metry} m", loc, pad=0.4, borderpad=0.5, sep=3,
                            frameon=True, size_vertical=metry / 30)
    pasek.patch.set_alpha(0.75)
    ax.add_artist(pasek)


def kamera(ax, kier):
    """Strzałka w rogu kadru: w którą stronę patrzy kamera."""
    dx, dy = KIERUNKI[kier]
    ax.annotate("", xy=(0.92 + 0.05 * dx, 0.88 + 0.05 * dy), xytext=(0.92 - 0.05 * dx, 0.88 - 0.05 * dy),
                xycoords="axes fraction", arrowprops=dict(arrowstyle="-|>", color="white", lw=2.5, mutation_scale=18))


def brak(ax, tekst):
    """Pusty panel z napisem, gdy dla kierunku nie ma arkusza."""
    ax.set_facecolor("0.93")
    ax.text(0.5, 0.5, tekst, ha="center", va="center", color="0.4", transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines[:].set_visible(False)


def widok(ax, b, kier, res=0.1):
    """Wycinek budynku b z kierunku kier z obrysem podstawy i dachu."""
    ax.set_title(f"{kier}: kamera patrzy na {NA[kier]}")
    if kier not in dostepne_kierunki(b):
        brak(ax, f"brak arkusza {kier}")
        return
    img, ext = czytaj(kier, kadr(b, kier), res)
    ax.imshow(img, extent=ext)
    obrys(ax, b.geometry, c=PODSTAWA, ls="--", label="podstawa")
    obrys(ax, dach(b, kier), c=DACH, label=f"dach (+{K * b.h:.0f} m)")
    ax.set_axis_off()
    skala(ax, 10)
    kamera(ax, kier)

## 1. Pliki

In [ ]:
opis_arkuszy().T

Każdy plik to ten sam teren widziany z innego kierunku (końcówka `_N`, `_E`, `_S`, `_W` w nazwie). Podgląd z piramid, 1 m/px:

In [ ]:
caly = zasieg().bounds
fig, axs = plt.subplots(2, 2, figsize=(15, 10), layout="constrained")
for ax, kier in zip(axs.flat, KIERUNKI):
    if kier not in KOLEJNOSC:
        brak(ax, f"brak arkusza {kier}")
        continue
    img, ext = czytaj(kier, caly, res=1.0)
    ax.imshow(img, extent=ext)
    ax.set_title(f"{kier}: kamera patrzy na {NA[kier]}, widać elewacje {ELEWACJE[kier]}")
    ax.set_axis_off()
    skala(ax, 100)
    kamera(ax, kier)
plt.show()

## 2. Jak działa ortofotomapa skośna

Zdjęcia są rektyfikowane na model terenu, więc grunt i podstawy budynków leżą na swoim miejscu. Dach na wysokości `h` przesuwa się od kamery o `K · h`, a między podstawą a dachem widać elewację.

In [ ]:
h, szer = 20, 14  # przykładowy budynek [m]
fig, ax = plt.subplots(figsize=(10, 3.8), layout="constrained")
ax.axhline(0, color="0.3", lw=1)
ax.add_patch(plt.Rectangle((0, 0), szer, h, fc="0.8", ec="0.3"))
for x in (0, szer):  # promienie przez krawędzie dachu; kamera daleko, więc są równoległe
    ax.annotate("", xy=(x + K * h, 0), xytext=(x - K * 12, h + 12),
                arrowprops=dict(arrowstyle="-|>", ls="--", color="tab:blue", lw=1.2))
ax.text(-K * 12 - 1, h + 12.5, "kamera patrzy na północ", color="tab:blue", ha="left", va="bottom")
pas = [(-14, 0, "grunt", "#d9cfa8"), (0, K * h, "elewacja\npołudniowa", "#e8915b"),
       (K * h, szer + K * h, "dach", "0.45"), (szer + K * h, 50, "grunt", "#d9cfa8")]
for a, b, nazwa, kolor in pas:  # co zapisze ortofotomapa w każdym punkcie gruntu
    ax.add_patch(plt.Rectangle((a, -4.5), b - a, 2.5, fc=kolor, ec="white"))
    ax.text((a + b) / 2, -3.25, nazwa, ha="center", va="center", fontsize=8,
            color="white" if kolor == "0.45" else "black")
ax.annotate("", xy=(0, -6.5), xytext=(szer, -6.5), arrowprops=dict(arrowstyle="|-|", color=PODSTAWA, lw=2))
ax.text(szer / 2, -8, "obrys budynku (podstawa)", ha="center", va="top", fontsize=8)
ax.annotate("", xy=(szer + K * h, h + 2), xytext=(szer, h + 2), arrowprops=dict(arrowstyle="->", color=DACH, lw=1.5))
ax.text(szer + K * h / 2, h + 3, f"przesunięcie = K·h = {K}·{h} = {K * h:.0f} m", ha="center", color=DACH)
ax.text(51, -3.25, "← co widać\n   na ortofotomapie", va="center", fontsize=8)
ax.set(xlim=(-16, 62), ylim=(-11, h + 16), xlabel="południe  ←   [m]   →  północ", aspect="equal")
ax.set_yticks([])
ax.spines[["left", "top", "right"]].set_visible(False)
plt.show()

Ten sam fragment z czterech kierunków. Po prawej nałożone dwa przeciwne widoki: grunt się pokrywa i wychodzi szary, dachy rozjeżdżają się w przeciwne strony.

In [ ]:
okno = (6433690, 5664790, 6433870, 5664910)  # 180 × 120 m wokół 12-kondygnacyjnego bloku
para = ("N", "S") if {"N", "S"} <= set(KOLEJNOSC) else tuple(KOLEJNOSC[:2])

fig, axs = plt.subplot_mosaic([["N", "E", "anaglif"], ["S", "W", "anaglif"]], figsize=(16, 7), layout="constrained",
                              width_ratios=[1, 1, 1.4])
obrazy = {}
for kier in KIERUNKI:
    if kier not in KOLEJNOSC:
        brak(axs[kier], f"brak arkusza {kier}")
        continue
    obrazy[kier], ext = czytaj(kier, okno, 0.2)
    axs[kier].imshow(obrazy[kier], extent=ext)
    axs[kier].set_title(kier)
    axs[kier].set_axis_off()
    kamera(axs[kier], kier)
szary = {k: obrazy[k].mean(2) for k in para}
axs["anaglif"].imshow(np.dstack([szary[para[0]], szary[para[1]], szary[para[1]]]).astype(np.uint8), extent=ext)
axs["anaglif"].set_title(f"{para[0]} (czerwony) + {para[1]} (turkusowy)")
axs["anaglif"].set_axis_off()
skala(axs["anaglif"], 20)
plt.show()

## 3. Obrysy budynków (BDOT10k)

Obrysy, funkcja i kondygnacje pochodzą z BDOT10k, a adres i zmierzona wysokość z OSM (pobieranego raz przez Overpass API do `data/`). Bez zmierzonej wysokości przyjmujemy `h = 3,3 · kondygnacje + 2 m`, a bez kondygnacji 8 m.

In [ ]:
bud = wczytaj_budynki()
print(f"{len(bud)} budynków na arkuszach; źródło wysokości:", bud.zrodlo_h.value_counts().to_dict())

In [ ]:
fig, axs = plt.subplot_mosaic([["mapa", "typy"], ["mapa", "kond"]], figsize=(15, 6), layout="constrained",
                              width_ratios=[2.6, 1])
img, ext = podklad(caly, res=1.0)
axs["mapa"].imshow(img, extent=ext)
bud.plot(ax=axs["mapa"], column="kondygnacje", cmap="viridis", alpha=0.6, edgecolor="white", lw=0.3,
         legend=True, legend_kwds={"label": "kondygnacje (BDOT10k)", "shrink": 0.7},
         missing_kwds={"facecolor": "none", "edgecolor": "white", "lw": 0.6})
axs["mapa"].set(title="Obrysy BDOT10k (białe kontury = brak liczby kondygnacji)", xlim=ext[:2], ylim=ext[2:])
axs["mapa"].set_axis_off()
skala(axs["mapa"], 100)

typy = bud.funkcja.value_counts().head(8)[::-1]
axs["typy"].barh(typy.index, typy.values, color="0.4")
axs["typy"].set(title="Przeważająca funkcja (BDOT10k)", xlabel="liczba budynków")
kond = bud.kondygnacje.value_counts().sort_index()
axs["kond"].bar(kond.index, kond.values, color="0.4")
axs["kond"].set(title="Liczba kondygnacji", xlabel="kondygnacje", ylabel="liczba budynków")
for nazwa in ("typy", "kond"):
    axs[nazwa].spines[["top", "right"]].set_visible(False)
plt.show()

## 4. Kalibracja przesunięcia dachu

Budynki ze zmierzoną wysokością: obrys podstawy i obrys przesunięty o `K · h`. We wszystkich kierunkach najlepiej pasuje K ≈ 0,85, czyli kamera odchylona o ok. 40° od pionu.

In [ ]:
KALIBRACJA = [101207542, 101195435, 101213933, 101189011]  # budynki z tagiem height w OSM
WARIANTY = [(0.70, "#00e5ff", ":"), (0.85, DACH, "-"), (1.00, "#d500f9", "-.")]

fig, axs = plt.subplots(len(KIERUNKI), len(KALIBRACJA), figsize=(16, 17), layout="constrained")
for j, osm_id in enumerate(KALIBRACJA):
    b = wybierz(bud, osm_id=osm_id)
    for i, kier in enumerate(KIERUNKI):
        ax = axs[i, j]
        ax.set_title(f"{kier} · OSM {osm_id} · h = {b.h:.1f} m")
        if kier not in dostepne_kierunki(b):
            brak(ax, f"brak arkusza {kier}")
            continue
        img, ext = czytaj(kier, kadr(b, kier, k=1.1), 0.1)
        ax.imshow(img, extent=ext)
        obrys(ax, b.geometry, c=PODSTAWA, ls="--", label="podstawa (obrys BDOT10k)")
        for k, kolor, ls in WARIANTY:
            obrys(ax, dach(b, kier, k), c=kolor, ls=ls, label=f"K = {k:.2f}")
        ax.set_axis_off()
        skala(ax, 10)
axs[0, 0].legend(loc="upper left", fontsize=8, framealpha=0.85)
plt.show()

## 5. Wybór budynku i widok z różnych stron

Kadr obejmuje podstawę i dach przesunięty o `K · h` w kierunku patrzenia, z marginesem 5 m.

In [ ]:
def pokaz_budynek(b):
    fig, axs = plt.subplot_mosaic([["mapa", "N", "E"], ["mapa", "S", "W"]], figsize=(16, 10), layout="constrained",
                                  width_ratios=[1.2, 1, 1])
    cx, cy = b.geometry.centroid.coords[0]
    img, ext = podklad((cx - 120, cy - 160, cx + 120, cy + 160), 0.5)
    axs["mapa"].imshow(img, extent=ext)
    obrys(axs["mapa"], b.geometry, c=DACH, lw=2.5)
    axs["mapa"].set_title("położenie (0,5 m/px)")
    axs["mapa"].set_axis_off()
    skala(axs["mapa"], 50)
    for kier in KIERUNKI:
        widok(axs[kier], b, kier)
    axs["N"].legend(loc="upper left", fontsize=8, framealpha=0.85)
    fig.suptitle(opis(b))
    plt.show()


pokaz_budynek(wybierz(bud, osm_id=101207542))

In [ ]:
pokaz_budynek(wybierz(bud, adres="Piastowska 36"))

## 6. Elewacja i liczba kondygnacji

Fasada zwrócona do kamery leży między krawędzią obrysu a tą krawędzią przesuniętą o `K · h`. Prostujemy ją do prostokąta w metrach. Okres rzędów okien bierzemy z autokorelacji pionowego profilu, a kondygnacje to `h / okres`. Gdy maksimum autokorelacji jest słabsze niż 0,4, okien nie widać (ściana szczytowa, cień, drzewa) i liczby nie podajemy.

In [ ]:
def fasady(b):
    """Wyprostowane fasady budynku b ze wszystkich dostępnych kierunków: {kier: (obraz, szerokość, okres, siła)}."""
    wynik = {}
    for kier in dostepne_kierunki(b):
        el = elewacja(b, kier)
        if el is not None:
            T, sila, *_ = okres_kondygnacji(el[0], b.h)
            wynik[kier] = (*el, T, sila)
    return wynik


def podpis(b, T, sila):
    return f"{b.h / T:.1f} kond." if sila >= 0.4 else "brak wyraźnych okien"


b = wybierz(bud, osm_id=101207542)
f = fasady(b)
fig, axs = plt.subplot_mosaic([list(f), ["ac"] * len(f)], figsize=(15, 9), layout="constrained",
                              width_ratios=[f[k][1] for k in f], height_ratios=[2, 1])
for kier, (el, L, T, sila) in f.items():
    axs[kier].imshow(el, extent=(0, L, 0, b.h))
    axs[kier].set(title=f"fasada z widoku {kier}: {podpis(b, T, sila)}", xlabel="szerokość [m]", ylabel="wysokość [m]")
    _, _, przes, ac = okres_kondygnacji(el, b.h)
    axs["ac"].plot(przes, ac, label=f"{kier}: maksimum {sila:.2f} przy {T:.2f} m")
axs["ac"].axvspan(2.4, 4.5, color="0.5", alpha=0.1, label="szukany okres kondygnacji")
axs["ac"].axhline(0.4, color="0.5", ls=":", lw=1)
axs["ac"].set(xlim=(0, 12), xlabel="przesunięcie w pionie [m]", ylabel="autokorelacja")
axs["ac"].legend(fontsize=8, ncols=2)
axs["ac"].spines[["top", "right"]].set_visible(False)
fig.suptitle(f"{opis(b)}")
plt.show()

In [ ]:
fig, axs = plt.subplots(len(KALIBRACJA), len(KIERUNKI), figsize=(15, 16), layout="constrained")
fig.suptitle("Fasady budynków ze zmierzoną wysokością (wiersz = budynek, kolumna = kierunek)")
for i, osm_id in enumerate(KALIBRACJA):
    b = wybierz(bud, osm_id=osm_id)
    f = fasady(b)
    for j, kier in enumerate(KIERUNKI):
        ax = axs[i, j]
        if kier not in f:
            brak(ax, f"brak fasady {kier}")
            continue
        el, L, T, sila = f[kier]
        ax.imshow(el, extent=(0, L, 0, b.h))
        ax.set_title(f"{kier}: {podpis(b, T, sila)}")
    axs[i, 0].set_ylabel(f"OSM {osm_id}\n{b.kondygnacje:.0f} kond. w BDOT10k, h = {b.h:.0f} m")
plt.show()

## 7. Galeria

In [ ]:
GALERIA = {"kamienica, Piastowska 36": 101211778, "blok, 7 kondygnacji": 101192162, "szkoła, Reja 3": 101202875,
           "przedszkole, Sienkiewicza 85": 101215270, "kaplica zakonna": 101212515, "garaż": 101190172}

fig, axs = plt.subplots(len(KIERUNKI), len(GALERIA), figsize=(17, 14), layout="constrained")
fig.suptitle("Wiersze: N, E, S, W")
for j, (etykieta, osm_id) in enumerate(GALERIA.items()):
    b = wybierz(bud, osm_id=osm_id)
    for i, kier in enumerate(KIERUNKI):
        widok(axs[i, j], b, kier)
        axs[i, j].set_title(f"{etykieta}\nh = {b.h:.0f} m ({b.zrodlo_h})" if i == 0 else "")
plt.show()

## 8. Wycinki dla agenta wizualnego

Na każdy budynek jeden katalog: wycinek z każdego kierunku, wyprostowana fasada i `meta.json` bez adresu i współrzędnych. Piksel 5 cm, bez obrysów. To samo robi `python -m ortofoto.wycinki <osm_id> ...`.

In [ ]:
KATALOG = WYNIKI / "wycinki"
EKSPORT = {"blok, 12 kondygnacji": 101207542, **GALERIA}
metadane = {osm_id: eksport_budynku(wybierz(bud, osm_id=osm_id), KATALOG) for osm_id in EKSPORT.values()}
print(json.dumps(metadane[101207542], ensure_ascii=False, indent=2))

In [ ]:
PLIKI = [*KIERUNKI, *(f"fasada_{k}" for k in KIERUNKI)]
POKAZ = [101207542, 101211778, 101215270]

fig, axs = plt.subplots(len(POKAZ), len(PLIKI), figsize=(18, 9), layout="constrained")
fig.suptitle(f"Pliki z {KATALOG.relative_to(WYNIKI.parent)}/<id budynku>/ wczytane z dysku")
for i, osm_id in enumerate(POKAZ):
    for j, nazwa in enumerate(PLIKI):
        ax = axs[i, j]
        plik = KATALOG / wybierz(bud, osm_id=osm_id).name / f"{nazwa}.jpg"
        if not plik.exists():
            brak(ax, "brak pliku")
            ax.set_title(f"{osm_id}/{nazwa}.jpg", fontsize=8)
            continue
        img = Image.open(plik)
        ax.imshow(img)  # dokładnie to, co dostanie agent
        ax.set_title(f"{osm_id}/{plik.name}\n{img.width} × {img.height} px", fontsize=8)
        ax.set_axis_off()
plt.show()